# Deploying Machine Learning Models in the Cloud

In M2 you trained a Titanic survival model and deployed a Gradio demo to Hugging Face Spaces. That demo was built for humans: it has sliders and buttons. In this notebook we build what companies actually run in production: a **prediction API** — a URL that other software can send data to and get predictions back. We will build the API with FastAPI, test it properly, pack it into a Docker container, and walk through deploying that exact container to the two biggest clouds: Google Cloud (GCP) and Amazon Web Services (AWS).

**What you will learn**

- What "the cloud" actually is, and why companies deploy models there
- The difference between a demo app (for humans) and an API (for software)
- How to build a prediction API with FastAPI and pydantic
- How to test an API in-process, before deploying anything
- What Docker containers are and why they solve "works on my machine"
- How to deploy the same container to Google Cloud Run and AWS App Runner
- How GCP and AWS services map to each other, and basic security and cost hygiene

## How to run this notebook

- **Locally:** open it in Jupyter (`jupyter lab` or `jupyter notebook`) and run cells top to bottom.
- **Google Colab:** upload the notebook (or open it from GitHub) and run cells top to bottom.

The setup cell below installs any missing packages, so both environments work out of the box. **Running this notebook requires no accounts at all** — every code cell runs on your machine (or Colab), including the API tests, which run in-process without starting a real server.

The cloud deployment sections (7 and 8) are **optional follow-along guides**. If you want to actually put your API on the internet, they require a free-tier Google Cloud or AWS account. Everything in them happens in your browser — nothing to install on your laptop.

In [1]:
# SETUP - run this cell first. It installs any missing packages.
# On Google Colab most packages are already installed, so this usually does nothing.
import importlib.util, subprocess, sys

def ensure(package, pip_name=None):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or package])

ensure("pandas"); ensure("numpy"); ensure("sklearn", "scikit-learn"); ensure("joblib"); ensure("fastapi"); ensure("httpx")

## 1. What "the cloud" actually is

**The cloud** is just computers you rent by the second, sitting in someone else's data center. A **data center** is a warehouse full of servers with fast internet, backup power, and staff who keep it all running. When you "deploy to the cloud", you copy your program onto one of those rented computers and it runs there instead of on your laptop.

Why do companies deploy models in the cloud instead of running them on a laptop?

- **Always on.** Your laptop sleeps, gets closed, goes on holiday. A cloud server runs 24/7.
- **Scales with demand.** If 10,000 users show up at once, the cloud can start more copies of your app automatically. Your laptop cannot.
- **Close to users.** Cloud providers have data centers all over the world, so responses arrive fast wherever your users are.
- **Pay only for use.** Modern cloud services bill per request or per second of compute. If nobody calls your API, you pay (almost) nothing.

The "big three" cloud providers:

| Provider | Usually called | Run by |
|---|---|---|
| Amazon Web Services | **AWS** | Amazon |
| Google Cloud Platform | **GCP** | Google |
| Microsoft Azure | **Azure** | Microsoft |

We cover the first two in this notebook. Azure works the same way with different names — more on that in section 9.

**How is this different from what we did in M2?** In M2 we deployed a *demo* to Hugging Face Spaces. That was the right tool for the job: a free, friendly page where a human can click around and see the model work. Today we build what companies actually run behind the scenes: an **API service in a real cloud**, the kind other software talks to millions of times per day.

## 2. From demo app to API

Our Gradio app has buttons and sliders. That is great for humans — but other **software cannot click buttons**. A mobile app, a website backend, or another team's service needs a way to ask your model for predictions programmatically. That way is an **API**.

**API (Application Programming Interface):** a URL that accepts data and returns answers. You send it a request, it sends back a response. No screens, no buttons — just data in, data out.

APIs usually exchange data as **JSON**:

- **JSON (JavaScript Object Notation):** a simple text format for structured data.
- It looks like a Python dictionary: `{"Age": 29, "Sex": "female"}`.
- Almost every programming language can read and write it, which is why APIs use it.

Here is the whole flow we are going to build:

```
 [ mobile app / website / other service ]
                  |
                  |  1. HTTP request:  POST /predict
                  |     {"Pclass": 1, "Sex": "female", "Age": 29, ...}
                  v
        +--------------------+
        |   our API service  |
        |  (FastAPI, in the  |
        |       cloud)       |
        |                    |
        |  2. model.predict  |
        +--------------------+
                  |
                  |  3. HTTP response:
                  |     {"prediction": 1, "probability": 0.93}
                  v
 [ mobile app / website / other service ]
```

The client sends passenger data as JSON, our service runs the model, and the answer comes back as JSON. That is the entire idea. Let's build it.

## 3. Train and save the model

Before we can serve predictions, we need a trained model. The cell below is a compact recap of everything you built in M2 — the same feature engineering from M2 notebook 1 ("01-data-prep-and-feature-engineering") and the tuned RandomForest settings from M2 notebook 3. If any step looks unfamiliar, revisit those notebooks; here we just run it in one go.

What the cell does:

1. Loads the Titanic data.
2. Engineers the M2 features: `Title` (from the name), `FamilySize`, `IsAlone`.
3. Builds one **pipeline** = preprocessing (imputing, scaling, one-hot encoding) + the RandomForest.
4. Trains it, prints test accuracy, and saves it with joblib.

We save the **whole pipeline**, not just the model, so the preprocessing travels with the model — the API can then accept raw passenger data and the pipeline handles the rest.

In [2]:
import os
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Load the data
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# 2. Feature engineering (same as M2 notebook 1)
# Title: the word before the dot in the name, e.g. "Braund, Mr. Owen" -> "Mr"
df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
common_titles = ["Mr", "Mrs", "Miss", "Master"]
df.loc[~df["Title"].isin(common_titles), "Title"] = "Rare"
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# 3. Features and target
numeric_features = ["Age", "Fare", "SibSp", "Parch", "FamilySize"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title"]
X = df[numeric_features + categorical_features]
y = df["Survived"]

# 4. Preprocessing: numbers get imputed + scaled, categories get imputed + one-hot encoded
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

# 5. Full pipeline: preprocessing + the tuned RandomForest from M2 notebook 3
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=3, random_state=42)),
])

# 6. Train and evaluate
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)
print(f"Test accuracy: {model.score(X_test, y_test):.3f}")

# 7. Save the WHOLE pipeline (preprocessing + model in one file)
os.makedirs("outputs", exist_ok=True)
model_path = "outputs/titanic_model.joblib"
joblib.dump(model, model_path)
print(f"Saved pipeline to {model_path}")

Test accuracy: 0.832
Saved pipeline to outputs/titanic_model.joblib


## 4. Build the prediction API with FastAPI

**FastAPI** is a modern Python library for building APIs. It is popular because you can get a working API in about 10 lines of code, and it generates interactive documentation for free.

First we describe what a valid request looks like, using **pydantic**:

- **pydantic:** a library that validates incoming data automatically.
- You define a class listing each field and its type (this is called an **input model**).
- FastAPI then checks every incoming request against it. Wrong types, missing fields, nonsense values — all rejected with a clear error message, for free. You never have to write `if not isinstance(...)` checks yourself.

Our `Passenger` model lists exactly the raw fields a client must send. The `= 0` style entries are defaults, so clients can skip those fields.

In [3]:
from pydantic import BaseModel

class Passenger(BaseModel):
    """The data a client must send to get a prediction."""
    Pclass: int          # ticket class: 1, 2 or 3
    Sex: str             # "male" or "female"
    Age: float           # age in years
    Fare: float          # ticket price
    SibSp: int = 0       # siblings/spouses aboard (default 0)
    Parch: int = 0       # parents/children aboard (default 0)
    Embarked: str = "S"  # port: "S", "C" or "Q" (default "S")

# Quick check: pydantic parses a dictionary into a validated object
example = Passenger(Pclass=1, Sex="female", Age=29, Fare=100.0)
print(example)

Pclass=1 Sex='female' Age=29.0 Fare=100.0 SibSp=0 Parch=0 Embarked='S'


Now the API itself. Two **endpoints** (an endpoint = one URL the API answers on):

- `GET /health` — returns `{"status": "ok"}`. Cloud platforms call this to check the service is alive.
- `POST /predict` — receives a `Passenger` as JSON, returns the prediction.

Inside `/predict` we rebuild the same engineered features the model was trained on (`Title`, `FamilySize`, `IsAlone`) — the same simple logic our M2 Gradio app used: title from sex and age, family size from the two family columns. Then we build a one-row DataFrame (the pipeline expects a DataFrame, exactly like in training), call the pipeline, and return the answer as JSON.

Note the naming: `GET` is for reading information, `POST` is for sending data in. That is a web convention you will see everywhere.

In [4]:
import joblib
import pandas as pd
from fastapi import FastAPI

app = FastAPI(title="Titanic Survival API")

# Load the trained pipeline once, at startup - not on every request
pipeline = joblib.load("outputs/titanic_model.joblib")

def derive_title(sex: str, age: float) -> str:
    # Same simple rule as our M2 Gradio app
    if sex == "female":
        return "Miss" if age < 18 else "Mrs"
    return "Master" if age < 13 else "Mr"

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(passenger: Passenger):
    family_size = passenger.SibSp + passenger.Parch + 1
    row = pd.DataFrame([{
        "Age": passenger.Age,
        "Fare": passenger.Fare,
        "SibSp": passenger.SibSp,
        "Parch": passenger.Parch,
        "FamilySize": family_size,
        "Pclass": passenger.Pclass,
        "Sex": passenger.Sex,
        "Embarked": passenger.Embarked,
        "Title": derive_title(passenger.Sex, passenger.Age),
    }])
    prediction = int(pipeline.predict(row)[0])
    probability = float(pipeline.predict_proba(row)[0][1])
    return {"prediction": prediction, "probability": round(probability, 3)}

print("API defined with 2 endpoints: GET /health and POST /predict")

API defined with 2 endpoints: GET /health and POST /predict


## 5. Test it before shipping

Professionals never deploy something they have not tested. FastAPI makes this easy with **TestClient**: a tool that *pretends to be the internet*. It sends requests to your app in-process — no server, no port, no network — and gives you back exactly the responses a real client would get. It works the same on your laptop, on Colab, anywhere.

First, the health check. A **status code** is a number every HTTP response carries: `200` means success.

In [5]:
from fastapi.testclient import TestClient

client = TestClient(app)

response = client.get("/health")
print("Status code:", response.status_code)
print("Body:", response.json())

Status code: 200
Body: {'status': 'ok'}


Now the real thing: predictions. We test two very different passengers:

- a 29-year-old woman travelling first class (historically, very likely to survive), and
- a 25-year-old man travelling third class (historically, very unlikely to survive).

The `probability` field is the model's confidence that the passenger survives: 0.9 means "9 chances out of 10".

In [6]:
# Passenger 1: first-class woman
first_class_woman = {"Pclass": 1, "Sex": "female", "Age": 29, "Fare": 100.0}
r1 = client.post("/predict", json=first_class_woman)
print("First-class woman :", r1.json())

# Passenger 2: third-class man
third_class_man = {"Pclass": 3, "Sex": "male", "Age": 25, "Fare": 7.9}
r2 = client.post("/predict", json=third_class_man)
print("Third-class man   :", r2.json())

First-class woman : {'prediction': 1, 'probability': 0.979}
Third-class man   : {'prediction': 0, 'probability': 0.083}


The woman gets `prediction: 1` (survives) with a high probability; the man gets `prediction: 0` (does not survive) with a low one. In plain words: the model is quite sure about both, and in opposite directions — exactly what we saw in M2.

One more professional habit: test that **bad input is rejected**. We send an age that is text instead of a number. Pydantic should refuse it with status code `422` (which means "I understood your request, but the data in it is invalid") — and we wrote zero validation code ourselves.

In [7]:
bad_passenger = {"Pclass": 1, "Sex": "female", "Age": "twenty-nine", "Fare": 100.0}
r3 = client.post("/predict", json=bad_passenger)
print("Status code:", r3.status_code)  # 422 = invalid input, rejected by pydantic
print("Error detail:", r3.json()["detail"][0]["msg"])
assert r3.status_code == 422
print("\nPydantic did its job: bad data never reached the model.")

Status code: 422
Error detail: Input should be a valid number, unable to parse string as a number

Pydantic did its job: bad data never reached the model.


All three tests pass: the service is alive, predictions make sense, and garbage input is rejected. **Test before you deploy** — it is much cheaper to find a bug on your laptop than in the cloud. Now let's ship it.

## 6. Containers — the shipping box

Here is a classic problem. Your API works on your laptop. You copy it to a server and it breaks: different Python version, different library versions, missing system packages. Every developer knows the phrase **"but it works on my machine!"**

**Docker** solves this. In plain words: Docker puts your app in a **box** that contains your code, Python itself, and the exact versions of every library — everything it needs. That box runs identically on any computer that has Docker: your laptop, a colleague's Windows PC, a Google server, an Amazon server.

Two words you need:

- **Image:** the recipe — a frozen snapshot of the box, built once from a file called a `Dockerfile`.
- **Container:** the cooked dish — a running copy of an image. You can run one container from an image, or a thousand.

Cloud services like Cloud Run and App Runner take an image and run containers from it. So our job is: write the recipe.

The next cells write a complete, self-contained deployment folder to `outputs/cloud-deployment/` with everything the cloud needs:

```
outputs/cloud-deployment/
    app.py                 <- the API (standalone version of section 4)
    titanic_model.joblib   <- the trained pipeline
    requirements.txt       <- exact libraries to install
    Dockerfile             <- the recipe for the image
    .dockerignore          <- files Docker should skip
```

First, `app.py` — the same API we built and tested above, as a standalone file. The only change: it loads the model from its own folder (using `__file__`, the path of the script itself), so it works no matter where the container runs it.

In [8]:
import os

deploy_dir = "outputs/cloud-deployment"
os.makedirs(deploy_dir, exist_ok=True)

app_py = '''"""Titanic Survival API - standalone version for cloud deployment."""
from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Titanic Survival API")

# Load the model from the same folder as this file
MODEL_PATH = Path(__file__).parent / "titanic_model.joblib"
pipeline = joblib.load(MODEL_PATH)


class Passenger(BaseModel):
    Pclass: int
    Sex: str
    Age: float
    Fare: float
    SibSp: int = 0
    Parch: int = 0
    Embarked: str = "S"


def derive_title(sex: str, age: float) -> str:
    if sex == "female":
        return "Miss" if age < 18 else "Mrs"
    return "Master" if age < 13 else "Mr"


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(passenger: Passenger):
    family_size = passenger.SibSp + passenger.Parch + 1
    row = pd.DataFrame([{
        "Age": passenger.Age,
        "Fare": passenger.Fare,
        "SibSp": passenger.SibSp,
        "Parch": passenger.Parch,
        "FamilySize": family_size,
        "Pclass": passenger.Pclass,
        "Sex": passenger.Sex,
        "Embarked": passenger.Embarked,
        "Title": derive_title(passenger.Sex, passenger.Age),
    }])
    prediction = int(pipeline.predict(row)[0])
    probability = float(pipeline.predict_proba(row)[0][1])
    return {"prediction": prediction, "probability": round(probability, 3)}
'''

with open(os.path.join(deploy_dir, "app.py"), "w") as f:
    f.write(app_py)
print("Wrote", os.path.join(deploy_dir, "app.py"))

Wrote outputs/cloud-deployment/app.py


Now the three remaining files, plus a copy of the trained model:

- `requirements.txt` — the exact libraries to install inside the box. We **pin scikit-learn to the version we trained with**: a model saved with one sklearn version may not load correctly with another.
- `Dockerfile` — the recipe (explained line by line right after the cell).
- `.dockerignore` — like `.gitignore`, but for Docker: files it should not copy into the image.

In [9]:
import shutil
import sklearn

# requirements.txt - pin scikit-learn to the exact training version.
# Why pin? A model saved with sklearn X may fail or behave differently under sklearn Y.
requirements = f"""fastapi
uvicorn
scikit-learn=={sklearn.__version__}
pandas
numpy
joblib
"""
with open(f"{deploy_dir}/requirements.txt", "w") as f:
    f.write(requirements)

# Dockerfile - the recipe for the image (every line explained below this cell)
dockerfile = """FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8080
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8080"]
"""
with open(f"{deploy_dir}/Dockerfile", "w") as f:
    f.write(dockerfile)

# .dockerignore - keep junk out of the image
with open(f"{deploy_dir}/.dockerignore", "w") as f:
    f.write("__pycache__/\n*.pyc\n.git/\n")

# Copy the trained pipeline into the deployment folder
shutil.copy("outputs/titanic_model.joblib", f"{deploy_dir}/titanic_model.joblib")

print("Deployment folder ready:")
for name in sorted(os.listdir(deploy_dir)):
    size_kb = os.path.getsize(os.path.join(deploy_dir, name)) / 1024
    print(f"  {name:22s} {size_kb:8.1f} KB")

Deployment folder ready:
  .dockerignore               0.0 KB
  Dockerfile                  0.2 KB
  __pycache__                 0.1 KB
  app.py                      1.4 KB
  requirements.txt            0.1 KB
  titanic_model.joblib     2219.7 KB


**The Dockerfile, line by line:**

- `FROM python:3.12-slim` — start from an official image that already contains Python 3.12; "slim" means a small version without extras we do not need.
- `WORKDIR /app` — create a folder called `/app` inside the box and work from there (like `cd /app`).
- `COPY requirements.txt .` — copy only the requirements file in first (Docker caches steps, so dependencies are not reinstalled every time the code changes).
- `RUN pip install --no-cache-dir -r requirements.txt` — install the exact library versions inside the box; `--no-cache-dir` keeps the image smaller.
- `COPY . .` — now copy everything else: `app.py` and `titanic_model.joblib`.
- `EXPOSE 8080` — document that the app listens on port 8080 (a **port** is a numbered door on a computer; 8080 is the door cloud services expect).
- `CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8080"]` — the command that runs when the container starts. **uvicorn** is the server program that actually runs FastAPI apps; `app:app` means "the object named `app` inside `app.py`"; `0.0.0.0` means "accept connections from outside the box", not just from inside.

That folder is our complete shippable unit. We will now deploy **the exact same folder** to two different clouds — without changing a single character in it. That is the whole point of Docker.

## 7. Deploy to Google Cloud — Cloud Run (optional follow-along)

**Cloud Run** is Google's "serverless containers" service. Serverless means: you never see or manage a server. You give Google a container, Google gives you back a public URL. It starts more containers when traffic grows and — the best part — **scales to zero**: when nobody is calling your API, no containers run and you pay nothing. The free tier is generous, which makes it ideal for student projects.

This section needs a Google account (free trial). Everything happens in the browser — nothing to install on your laptop.

**Follow along:**

1. **Create a Google Cloud account** at [cloud.google.com](https://cloud.google.com) and activate the free trial. You get $300 in credits for 90 days. It asks for a card to verify you are human, but it does **not** auto-charge when credits run out — you would have to upgrade manually.

2. **Open Cloud Shell** at [shell.cloud.google.com](https://shell.cloud.google.com). This is a small Linux terminal running in your browser, with `gcloud` (Google's command-line tool) and `docker` preinstalled and already authenticated as you. Nothing to install, no credentials to copy anywhere.

3. **Upload the deployment folder.** In Cloud Shell, click the three-dot menu, then "Upload", and upload the contents of your `outputs/cloud-deployment/` folder (the five files). Then `cd` into the folder you uploaded them to.

4. **Deploy with one command:**

   ```bash
   gcloud run deploy titanic-api --source . --region europe-west4 --allow-unauthenticated
   ```

   What each part means:
   - `gcloud run deploy titanic-api` — create (or update) a Cloud Run service named `titanic-api`.
   - `--source .` — build the container from the current folder; Google finds your Dockerfile, builds the image, and stores it for you.
   - `--region europe-west4` — run it in Google's Netherlands data center, close to Fontys, so responses are fast.
   - `--allow-unauthenticated` — make the URL public, so anyone can call it (fine for a class project; see section 10).

   The first deploy takes a few minutes. At the end it prints: `Service URL: https://titanic-api-....run.app`

5. **Test your live API.** Replace the URL with yours and paste this into Cloud Shell (or any terminal):

   ```bash
   curl -X POST "https://YOUR-SERVICE-URL.run.app/predict" \
        -H "Content-Type: application/json" \
        -d '{"Pclass": 1, "Sex": "female", "Age": 29, "Fare": 100.0}'
   ```

   You should get back `{"prediction": 1, "probability": ...}` — the same answer your TestClient got, but now from a real server on the internet. Your model has a public URL.

6. **View logs.** In the Cloud Console, go to Cloud Run, click `titanic-api`, then the "Logs" tab. Every request, every error, every `print()` from your app shows up here. This is where you look first when something misbehaves.

7. **CLEAN UP when you are done:**

   ```bash
   gcloud run services delete titanic-api --region europe-west4
   ```

   **Always clean up cloud resources when you finish an exercise.** Also set a budget alert on day one: in the Cloud Console go to Billing, then Budgets and alerts, and create a budget of e.g. 5 EUR with an email alert. It takes two minutes and removes all surprise-bill anxiety.

**Cost reality check:** Cloud Run's free tier includes millions of requests per month, and the service scales to zero between requests. For a student project like this API, the realistic cost is **effectively 0 EUR** — even without the trial credits.

## 8. Deploy to AWS — App Runner (optional follow-along)

**App Runner** is the AWS equivalent of Cloud Run: same idea — container in, URL out. The workflow has a few more steps than on GCP (AWS makes you store the image yourself first), but the result is identical. And here is the key observation: we deploy **the exact same folder, unchanged**. That is Docker keeping its promise.

Along the way you will meet **ECR (Elastic Container Registry)**: AWS's storage service for container images — a place to push your image so AWS services can pull and run it.

**Follow along:**

1. **Create an AWS account** at [aws.amazon.com](https://aws.amazon.com) (free tier). Like Google, it asks for a card for verification.

2. **Open AWS CloudShell**: log in to the AWS Console, pick a region near you (e.g. `eu-central-1`, Frankfurt) in the top-right corner, and click the terminal icon in the top bar. Like Google's Cloud Shell, it is a browser terminal with the `aws` command-line tool preinstalled and already authenticated as you.

3. **Upload the folder.** In CloudShell: Actions, then "Upload file". CloudShell uploads one file at a time, so zip the folder first on your machine (`cloud-deployment.zip`), upload the zip, then run `unzip cloud-deployment.zip && cd cloud-deployment`.

4. **Create an ECR repository** (a named shelf for your image):

   ```bash
   aws ecr create-repository --repository-name titanic-api
   ```

   The output includes a `repositoryUri` that looks like `123456789012.dkr.ecr.eu-central-1.amazonaws.com/titanic-api`. That number is your AWS account ID. Use YOUR uri in the commands below.

5. **Build the image and push it to ECR:**

   ```bash
   # 1. Log Docker in to your ECR (safe: your CLI session provides the auth, no passwords typed)
   aws ecr get-login-password | docker login --username AWS --password-stdin YOUR_REPOSITORY_URI

   # 2. Build the image from the Dockerfile in this folder, name it titanic-api
   docker build -t titanic-api .

   # 3. Tag it (give it a second name that says where it will live in ECR)
   docker tag titanic-api:latest YOUR_REPOSITORY_URI:latest

   # 4. Push (upload) it to ECR
   docker push YOUR_REPOSITORY_URI:latest
   ```

   Line by line: log in to your image storage, cook the recipe into an image, label the box with its destination address, ship it to the warehouse.

   Note: if `docker build` complains about disk space in CloudShell, you can instead run these same commands on any machine with Docker installed — the point is that they are the same everywhere.

6. **Create the App Runner service** (in the AWS Console, search "App Runner"):
   - Click **Create service**.
   - Source: **Container registry**, then **Amazon ECR**, then browse to `titanic-api:latest`.
   - Deployment settings: manual is fine; let it create the suggested ECR access role.
   - Service name: `titanic-api`. **Port: 8080** (this must match our Dockerfile's `EXPOSE 8080`).
   - Instance size: the **smallest** (0.25 vCPU, 0.5 GB) — plenty for this model.
   - Click **Create and deploy**. After a few minutes you get a URL like `https://xxxxxxxx.eu-central-1.awsapprunner.com`.

7. **Test it** with the same `curl` command from section 7, pointing at your App Runner URL. Same container, same request, same answer — different cloud.

8. **CLEAN UP — important:** App Runner does **not** scale to zero on the same terms as Cloud Run: a paused-but-existing service and stored images can still cost money. When you are done:
   - In App Runner: select the service, then **Actions, Delete**.
   - In ECR: open the `titanic-api` repository, delete the images, then delete the repository.

**Honest comparison:** AWS took about eight steps where Google took one command. Same container, same result. Different clouds package the same idea with different amounts of assembly required — which is exactly why the industry standardized on containers: your app does not care.

## 9. The bigger picture

You now know two clouds. The good news: they offer the same *shapes* of service under different names. Learn the shape once and you can find it anywhere:

| What it does | GCP name | AWS name |
|---|---|---|
| Run a container, get a URL | **Cloud Run** | **App Runner** |
| Run a single function on demand | Cloud Functions | Lambda |
| Full ML platform | Vertex AI | SageMaker |
| Store files | Cloud Storage | S3 |

**When do you graduate to Vertex AI / SageMaker?** When the model itself becomes a full-time job. These platforms add: **managed training** (they run and track your training jobs on rented hardware), a **model registry** (versioned storage of every model you have trained, so you know exactly what is in production), **monitoring** (alerts when the incoming data starts looking different from the training data), and **autoscaling endpoints** (prediction APIs that grow and shrink with traffic automatically). For one RandomForest and a class project, Cloud Run is the right tool; for a team retraining models weekly, those platforms earn their complexity.

**And Azure?** Microsoft's cloud has the same shapes with its own names: Container Apps plays the role of Cloud Run / App Runner, and Azure ML plays the role of Vertex AI / SageMaker. If you understand this notebook, you can navigate Azure too.

## 10. Security and cost hygiene

Four rules to remember for the rest of your career:

1. **Never put credentials in code, notebooks, or git.** No API keys, no passwords, no tokens — not even "just for testing". Notice that this entire notebook contains zero secrets: the `gcloud` and `aws` commands work because *you* logged in on *your* machine (`gcloud auth login` / `aws configure` in your own terminal or browser shell), and the CLI handles authentication from there. That is the professional pattern.
2. **`--allow-unauthenticated` means PUBLIC.** Anyone on the internet can call your API. Fine for a class project with a Titanic model; not fine for real data. Real companies put authentication in front of the API (API keys, tokens, or a login system) so only approved clients can call it.
3. **Set budget alerts before deploying anything.** Two minutes of work, sleep guaranteed.
4. **Delete what you stop using.** The cloud bills while you sleep. Finished the exercise? Delete the service, delete the images.

## 11. Where this goes next

Look at the system view from notebook 1:

```
[data] -> [training] -> [model file] -> [API service] -> [applications]
                                        ^^^^^^^^^^^^^
                                        you just built this
```

Your model no longer lives in a notebook — it lives at a URL, always on, ready to answer any software that asks. The last box is still empty, though: who actually *calls* this API? In notebook 3 we close the loop and build **applications that consume your API** — turning predictions into something a user actually sees.

## Key takeaways

- The cloud is rented computers in data centers: always on, scales with demand, close to users, pay per use.
- Demos (Gradio) are for humans; **APIs** are for software: JSON in, JSON out over HTTP.
- **FastAPI** builds APIs in a few lines; **pydantic** validates every incoming request for free (bad input gets a 422, never reaches the model).
- Save the **whole pipeline** so preprocessing travels with the model.
- **Test before deploying** with `TestClient` — in-process, no server needed.
- **Docker** packs app + Python + exact library versions into an image that runs identically anywhere; a container is a running copy of an image.
- **Cloud Run** (GCP) and **App Runner** (AWS) both do: container in, URL out. Cloud Run scales to zero; App Runner should be deleted when idle.
- Same shapes, different names, across all clouds (Cloud Run/App Runner, Cloud Functions/Lambda, Vertex AI/SageMaker, Cloud Storage/S3).
- Never commit credentials; public endpoints are really public; set budget alerts; delete what you stop using.

## Practice exercises

Try these on your own before looking at the solutions.

**Exercise 1 — Model info endpoint.** APIs in production usually expose an endpoint that reports what model is running. Add a `GET /model-info` endpoint to the app that returns the model type and the scikit-learn version it was trained with. Test it with `TestClient`.

**Exercise 2 — Reject impossible ages.** Our `Passenger` model happily accepts `Age = -5`. Pydantic's `Field(ge=0)` sets a rule: the value must be **g**reater than or **e**qual to 0. Create a stricter passenger model that rejects negative ages, wire it to a new endpoint, and prove with a test that a negative age gets status code 422.

**Exercise 3 — Deploy somewhere else.** A teammate in the US wants the API close to their users. Write the single `gcloud` command that deploys the same folder as a service named `titanic-api-us` in the region `us-central1`, publicly reachable.

**Exercise 4 — Pick the right service.** For each scenario, choose **Cloud Run**, **Lambda** (Cloud Functions on GCP), or **SageMaker** (Vertex AI on GCP), and justify in one sentence:
   - (a) A startup wants to serve a containerized prediction API with unpredictable traffic and a tiny budget.
   - (b) A team needs to run a 20-line Python function that resizes an image every time a file is uploaded.
   - (c) A bank retrains fraud models weekly, must track every model version, and needs alerts when input data drifts.

## Solutions

**Solution 1 — Model info endpoint.** We add the endpoint to our existing `app` and test it. Returning the sklearn version matters in real life: it tells you instantly whether the serving environment matches the training environment.

In [10]:
import sklearn

@app.get("/model-info")
def model_info():
    return {
        "model_type": type(pipeline.named_steps["classifier"]).__name__,
        "sklearn_version": sklearn.__version__,
        "n_trees": pipeline.named_steps["classifier"].n_estimators,
    }

r = client.get("/model-info")
print("Status code:", r.status_code)
print("Body:", r.json())
assert r.status_code == 200

Status code: 200
Body: {'model_type': 'RandomForestClassifier', 'sklearn_version': '1.9.0', 'n_trees': 300}


**Solution 2 — Reject impossible ages.** `Field(ge=0)` adds the rule to the type. We define a stricter model, expose it on a new endpoint `/predict-strict`, and prove that pydantic rejects a negative age with a 422 — and accepts a valid one.

In [11]:
from pydantic import Field

class StrictPassenger(BaseModel):
    Pclass: int
    Sex: str
    Age: float = Field(ge=0)   # ge=0 -> "greater than or equal to 0"
    Fare: float
    SibSp: int = 0
    Parch: int = 0
    Embarked: str = "S"

@app.post("/predict-strict")
def predict_strict(passenger: StrictPassenger):
    return predict(Passenger(**passenger.model_dump()))

# Negative age -> rejected with 422
r_bad = client.post("/predict-strict",
                    json={"Pclass": 1, "Sex": "female", "Age": -5, "Fare": 100.0})
print("Negative age -> status:", r_bad.status_code)
print("Error:", r_bad.json()["detail"][0]["msg"])
assert r_bad.status_code == 422

# Valid age -> still works
r_ok = client.post("/predict-strict",
                   json={"Pclass": 1, "Sex": "female", "Age": 29, "Fare": 100.0})
print("Valid age    -> status:", r_ok.status_code, "body:", r_ok.json())
assert r_ok.status_code == 200

Negative age -> status: 422
Error: Input should be greater than or equal to 0
Valid age    -> status: 200 body: {'prediction': 1, 'probability': 0.979}


**Solution 3 — Deploy somewhere else.**

```bash
gcloud run deploy titanic-api-us --source . --region us-central1 --allow-unauthenticated
```

Only two things changed versus section 7: the service name (`titanic-api-us`) and the region (`us-central1`, Iowa, USA). The folder — and the container built from it — is exactly the same. You would run this from inside the `cloud-deployment` folder in Cloud Shell, and remember to delete this service too when done.

**Solution 4 — Pick the right service.**

- **(a) Cloud Run.** It runs any container, scales to zero when idle (tiny budget) and scales up automatically when traffic spikes (unpredictable traffic).
- **(b) Lambda / Cloud Functions.** A single small function triggered by an event (file upload) is exactly what function services are for — no container or API needed.
- **(c) SageMaker / Vertex AI.** Weekly managed retraining, a model registry for version tracking, and data-drift monitoring are precisely the extra features these ML platforms add.